In [2]:
import pickle as pkl
import numpy as np
from pathlib import Path
import tarfile
import glob
import re
from collections import defaultdict, Counter
import itertools
import matplotlib.pyplot as plt
import pandas as pd
# import dionysus

# from helper_functions import *
# from KK_zz_apex_LS import *

from hierarchicalsoftmax import SoftmaxNode, HierarchicalSoftmaxLoss, HierarchicalSoftmaxLinear
from hierarchicalsoftmax.inference import (
    greedy_predictions,
    node_probabilities,
)

import numpy as np
from collections import defaultdict, Counter
import itertools
import pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim


/Users/levisvaren/anaconda3/envs/transformers/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load the pickle files from Kritika and save for zigzag PH

{sequenc: {'sequence_id':id, 'mapped_label':label}}

Stratified 5-fold split based on superfamily labels
* if only order provided, then superfamily==order for sake of split

<80bp length removed

'>5% non atgc removed

**Should also remove if n superfamily < 5**

In [3]:
def process_pkl(mapped_data):
    # Save sequence and label
    sequences = []
    labels = []
    for k, value in mapped_data.items():
        sequences.append(k.lower())
        labels.append(value['mapped_label'])

    seq_x = [re.sub(r'[^atcg]', 'x', seq) for seq in sequences]

    # Create df, split labels into order and superfamily
    seq_df = pd.DataFrame({'sequence':sequences, 'seq_x':seq_x, 'labels':labels})
    seq_df[['order', 'superfamily']] = seq_df['labels'].str.split('/',expand=True) #split
    seq_df['superfamily'] = seq_df['superfamily'].fillna(seq_df['order']) # if only order, superfamily==order
    return seq_df, sequences, labels

def kfold_strat_split(seq_df, k=5, seed=7):
    # Split 
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    X = np.array(list(seq_df['seq_x']))
    # y = np.array(labels)
    y = np.array(list(seq_df['superfamily']))

    for fold_index, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        col_name = f'fold_{fold_index}'
        seq_df[col_name] = 'train' # Default everyone to train
        seq_df.loc[val_idx, col_name] = 'test'

    return seq_df

def kfold_split(seq_df, k=5, shuffle=True, seed=7):
    # Split 
    skf = KFold(n_splits=k, shuffle=True, random_state=seed)
    X = np.array(list(seq_df['seq_x']))
    y = np.array(list(seq_df['labels']))
    # y = np.array(list(seq_df['superfamily']))

    for fold_index, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        col_name = f'fold_{fold_index}'
        seq_df[col_name] = 'train' # Default everyone to train
        seq_df.loc[val_idx, col_name] = 'test'

    return seq_df

def get_seq_stats(seq_df):
    seq_df['seq_length'] = seq_df['seq_x'].str.len() # seq length
    seq_df['n_non_actg'] = seq_df['seq_x'].str.count('[^actg]') # number non canonical
    seq_df['frac_non_actg'] = seq_df['n_non_actg'] / seq_df['seq_length'] # fraction non canonical
    return seq_df

def filter_seqs(seq_df, len_thresh=80, frac_thresh=0.2): #kritika already did the len_thresh
    print(f"Original shape: {seq_df.shape}")
    seq_df_filt = seq_df[(seq_df['seq_length'] >= len_thresh) & (seq_df['frac_non_actg'] <= frac_thresh)]
    # Remove any labels with <5 sequences
    label_counts = Counter(seq_df_filt['labels'])
    to_remove = {label for label, count in label_counts.items() if count < 5}
    seq_df_filt = seq_df_filt[~seq_df_filt['labels'].isin(to_remove)]
    print(f"Removing these labels: {to_remove}")
    print(f"New shape: {seq_df_filt.shape}")
    return seq_df_filt.reset_index()

def save_fasta(filename, seq_df):
    sequences = list(seq_df['seq_x'])
    labels = list(seq_df['labels'])
    with open(filename, "w") as f:
        for i, (label, seq) in enumerate(zip(labels, sequences)):
            f.write(f">{i}_{label}\n")
            f.write(f"{seq.lower()}\n")
    print(f"Saved here: {filename}")

def split_fasta(input_file, path, chunk_size=1000):
    with open(input_file, 'r') as f:
        sequences = []
        current_seq = []
        file_count = 1
        
        for line in f:
            line = line.strip()
            if not line: continue
            
            if line.startswith('>'):
                # If we have a sequence buffered, join it and store it
                if current_seq:
                    sequences.append("".join(current_seq))
                    current_seq = []
                
                # Check if we need to save the current chunk
                if len(sequences) >= chunk_size:
                    save_chunk(sequences, path, file_count)
                    sequences = []
                    file_count += 1
            else:
                current_seq.append(line)

        # Handle the very last sequence in the file
        if current_seq:
            sequences.append("".join(current_seq))
        
        # Save the final remaining chunk
        if sequences:
            save_chunk(sequences, path, file_count)

def save_chunk(data, path, count):
    output_name = f"{path}/chunk_{count}.txt"
    with open(output_name, 'w') as out:
        out.write("\n".join(data) + "\n")
    # print(f"Saved {output_name}")

In [16]:
all_labels = []

### Repbase

In [40]:
# Load data
pkl_path = "./../TERRIER_labellingsystems_alldatasets"
dataset = "RepBase31pt04_converted2_terrierlabelsMay13.pkl"
db_path = f"{pkl_path}/{dataset}"

filename = '../repbase_QC_051326.fasta'
save_path = '../Repbase/'

csv_save = '../repbase_QC_051326.csv'

with open(db_path, "rb") as f:
    mapped_data = pkl.load(f)

seq_df, sequences, labels = process_pkl(mapped_data)
all_labels.extend(labels)

# Filter
seq_df = get_seq_stats(seq_df)
seq_df = filter_seqs(seq_df, len_thresh=80, frac_thresh=0.2)

# Stratified split
seq_df = kfold_strat_split(seq_df, k=5, seed=7)
# seq_df = kfold_split(seq_df)

# Save df 
seq_df['dataset'] = 'Repbase'
seq_df.to_csv(csv_save, index=False)

# # Save big fasta
# save_fasta(filename, seq_df)

# # Save chunks
# output_dir = Path(save_path)
# output_dir.mkdir(parents=True, exist_ok=True)
# split_fasta(filename, save_path, 5000)


Original shape: (116787, 8)
Removing these labels: set()
New shape: (116766, 8)


In [4]:
seq_df['labels'].value_counts()

labels
LTR/Gypsy           36310
LTR/Copia           11906
LTR/ERV              9442
LTR/Pao              8566
DNA/hAT              7672
LINE/L1              5429
DNA/TcMar            5204
DNA/Harbinger        2812
DNA/MULE             2758
SINE/tRNA            2547
DNA                  2371
RC/Helitron          2137
LTR/DIRS             2028
LINE/RTE             1762
DNA/CMC              1740
PLE                  1521
LINE/CR1             1334
LINE/I               1131
LINE/L2              1088
LTR                  1074
DNA/Kolobok           972
DNA/PiggyBac          701
LINE/Tad1             689
DNA/Academ            652
Satellite             574
LINE                  444
LINE/R1               419
DNA/Sola              413
DNA/P                 348
LINE/R2               345
DNA/Crypton           296
DNA/Maverick          264
LTR/Caulimovirus      216
SINE/7SL              192
DNA/Merlin            187
LINE/Rex-Babar        180
DNA/Dada              169
DNA/IS3EU             126
DNA/Z

### RepetDB

In [42]:
# Load data
pkl_path = "./../TERRIER_labellingsystems_alldatasets"
dataset = "Repetdb_allfiltered_terrier_May13.pkl"
db_path = f"{pkl_path}/{dataset}"

filename = '../repetdb_QC_051326.fasta'
save_path = '../RepetDB/'

csv_save = '../repetdb_QC_051326.csv'

with open(db_path, "rb") as f:
    mapped_data = pkl.load(f)

seq_df, sequences, labels = process_pkl(mapped_data)
all_labels.extend(labels)

# Filter
seq_df = get_seq_stats(seq_df)
seq_df = filter_seqs(seq_df, len_thresh=80, frac_thresh=0.2)

# Stratified split
seq_df = kfold_strat_split(seq_df, k=5, seed=7)
# seq_df = kfold_split(seq_df)

# Save df 
seq_df['dataset'] = 'RepetDB'
seq_df.to_csv(csv_save, index=False)

# # Save big fasta
# save_fasta(filename, seq_df)

# # Save chunks
# output_dir = Path(save_path)
# output_dir.mkdir(parents=True, exist_ok=True)
# split_fasta(filename, save_path, 5000)

Original shape: (73155, 8)
Removing these labels: {'RC/Helitron'}
New shape: (73153, 8)


In [6]:
seq_df['labels'].value_counts()

labels
LTR/Gypsy        46710
LTR/Copia        25444
DNA/hAT            386
DNA/Harbinger      271
DNA/TcMar          206
DNA/CMC             98
LINE/I              38
Name: count, dtype: int64

### MnTEdb

In [43]:
# Load data
pkl_path = "./../TERRIER_labellingsystems_alldatasets"
dataset = "MnTEdb_terrier.pkl"
db_path = f"{pkl_path}/{dataset}"

filename = '../mntedb_QC_051326.fasta'
save_path = '../MnTEdb/'

csv_save = '../mntedb_QC_051326.csv'

with open(db_path, "rb") as f:
    mapped_data = pkl.load(f)

seq_df, sequences, labels = process_pkl(mapped_data)
all_labels.extend(labels)

# Filter
seq_df = get_seq_stats(seq_df)
seq_df = filter_seqs(seq_df, len_thresh=80, frac_thresh=0.2)

# Stratified split
seq_df = kfold_strat_split(seq_df, k=5, seed=7)
# seq_df = kfold_split(seq_df)

# Save df 
seq_df['dataset'] = 'MnTEdb'
seq_df.to_csv(csv_save, index=False)

# # Save big fasta
# save_fasta(filename, seq_df)

# # Save chunks
# output_dir = Path(save_path)
# output_dir.mkdir(parents=True, exist_ok=True)
# split_fasta(filename, save_path, 5000)

Original shape: (5372, 8)
Removing these labels: set()
New shape: (4927, 8)


In [8]:
seq_df['labels'].value_counts()

labels
LTR/Copia        1426
LTR/Gypsy        1298
DNA/hAT          1041
LTR               840
DNA/Harbinger     272
RC/Helitron        31
LINE/L1            19
Name: count, dtype: int64

In [ ]:
all_labels = list(set(all_labels))

['DNA/Harbinger',
 'DNA/Maverick',
 'SINE/tRNA',
 'SINE',
 'DNA/Merlin',
 'DNA/TcMar',
 'DNA/P',
 'LINE',
 'DNA/Ginger',
 'DNA/IS3EU',
 'LTR/Copia',
 'DNA/PiggyBac',
 'LTR/Caulimovirus',
 'DNA',
 'LINE/Rex-Babar',
 'DNA/CMC',
 'Satellite',
 'DNA/Crypton',
 'DNA/hAT',
 'DNA/Zator',
 'LINE/Dong-R4',
 'DNA/Sola',
 'LINE/Tad1',
 'SINE/U',
 'LINE/L1',
 'SINE/5S',
 'LTR/Gypsy',
 'LINE/R1',
 'DNA/Zisupton',
 'LINE/Proto2',
 'LINE/L2',
 'LTR',
 'DNA/Kolobok',
 'LINE/Dualen',
 'Structural_RNA',
 'DNA/Dada',
 'DNA/Novosib',
 'DNA/MULE',
 'LTR/Pao',
 'LINE/I',
 'Other',
 'DNA/Academ',
 'LINE/Proto1',
 'LTR/DIRS',
 'LINE/R2',
 'RC/Helitron',
 'SINE/7SL',
 'LINE/CR1',
 'LINE/RTE',
 'LINE/CRE',
 'LTR/ERV',
 'PLE']

In [31]:
ORDER_TO_SUPERFAMILIES = defaultdict(list)

In [ ]:
for l in all_labels:
    order_super = l.split('/')
    if len(order_super) == 2:
        ORDER_TO_SUPERFAMILIES[order_super[0]].append(order_super[1])
    elif len(order_super) == 1:
        ORDER_TO_SUPERFAMILIES[order_super[0]].append('')
for order, super in ORDER_TO_SUPERFAMILIES.items():
    ORDER_TO_SUPERFAMILIES[order] = list(set(super))

ORDER_TO_SUPERFAMILIES

In [44]:
pd.read_csv('../mntedb_QC_051326.csv')


,index,sequence,seq_x,labels,order,superfamily,seq_length,n_non_actg,frac_non_actg,fold_0,fold_1,fold_2,fold_3,fold_4,dataset
0,0,tgttgatgtcatagctgactgctgacctagtgtaattagtggcaga...,tgttgatgtcatagctgactgctgacctagtgtaattagtggcaga...,LTR/Copia,LTR,Copia,4872,0,0.0,train,train,train,test,train,MnTEdb
1,1,tgtgaaaaattgagaaagaactagaatatttctcattgaatcatga...,tgtgaaaaattgagaaagaactagaatatttctcattgaatcatga...,LTR/Copia,LTR,Copia,5007,0,0.0,train,test,train,train,train,MnTEdb
2,2,tgtcagaaatataggagtagataggtttagatatggtttatgcgtt...,tgtcagaaatataggagtagataggtttagatatggtttatgcgtt...,LTR/Copia,LTR,Copia,5026,0,0.0,train,train,test,train,train,MnTEdb
3,3,tgttgagatttccaaatcaattagggaagattcctaagaattatcc...,tgttgagatttccaaatcaattagggaagattcctaagaattatcc...,LTR/Copia,LTR,Copia,5238,0,0.0,train,train,train,test,train,MnTEdb
4,4,tggaggagaaaaacaatgtgaactttgctattattaatcagaaaac...,tggaggagaaaaacaatgtgaactttgctattattaatcagaaaac...,LTR/Copia,LTR,Copia,9961,0,0.0,train,test,train,train,train,MnTEdb
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4922,5367,tgttaggtatttttattagttggcaatttattggcaatcttgtaaa...,tgttaggtatttttattagttggcaatttattggcaatcttgtaaa...,LTR,LTR,LTR,4230,0,0.0,train,train,test,train,train,MnTEdb
4923,5368,tgatgtagctcctgctaccagaaaggagagataatttttatcttta...,tgatgtagctcctgctaccagaaaggagagataatttttatcttta...,LTR,LTR,LTR,5262,0,0.0,train,train,train,test,train,MnTEdb
4924,5369,tctttgagagtacgagagttgtagttagccaacttgtctacatcaa...,tctttgagagtacgagagttgtagttagccaacttgtctacatcaa...,LTR,LTR,LTR,6871,0,0.0,train,train,train,test,train,MnTEdb
4925,5370,tgttaggactgtttttgttcctgatgtcacagagagtgttataaca...,tgttaggactgtttttgttcctgatgtcacagagagtgttataaca...,LTR,LTR,LTR,5404,0,0.0,test,train,train,train,train,MnTEdb


In [38]:
ORDER_TO_SUPERFAMILIES = {
    'LTR': ['','ERV','Pao','Gypsy','DIRS','Caulimovirus','Copia'],
    'DNA': ['','PiggyBac','Ginger','Academ','Crypton','Sola','CMC',
              'IS3EU','Novosib','Harbinger','P','hAT','Maverick',
              'Dada','Zator','Kolobok','Merlin','MULE','Zisupton','TcMar'],
    'LINE': ['','CRE','Proto1','Rex-Babar','L2','Tad1','Proto2','I',
              'RTE','Dualen','Dong-R4','CR1','R2','L1','R1'],
    'Satellite': [''],
    'RC': ['Helitron'],
    'SINE': ['', 'U', '7SL', 'tRNA', '5S'],
    'Structural_RNA': [''],
    'PLE': [''],
    'Other': ['']}

In [ ]:
superfamily_colors_new = {
    # LTR-like / retroviral group: blues
    "Pao": "#eff3ff",          # old Bel-Pao
    "Copia": "#bdd7e7",
    "ERV": "#6baed6",
    "Gypsy": "#2171b5",
    "DIRS": "#ffff33",         # kept from old order_colors
    "Caulimovirus": "#084594",
 
    # DNA / former TIR-like group: oranges/browns
    "CMC": "#feedde",          # old CACTA-like color
    "TcMar": "#fdbe85",
    "hAT": "#fd8d3c",
    "MULE": "#e6550d",         # old MuLE
    "Harbinger": "#a63603",    # old PIF
    
    # New DNA superfamilies: related orange/brown shades
    "P": "#fdd0a2",
    "PiggyBac": "#fdae6b",
    "Zator": "#e34a33",
    "Merlin": "#b30000",
    "Kolobok": "#7f2704",
    "Maverick": "#d94801",
    "Novosib": "#8c2d04",
    "Zisupton": "#cc4c02",
    "Crypton": "#993404",
    "Academ": "#ec7014",
    "IS3EU": "#fe9929",
    "Dada": "#d95f0e",
    "Sola": "#f16913",
    "Ginger": "#a63603",
 
    # LINE group: purples
    "CR1": "#f2f0f7",
    "I": "#dadaeb",
    "L1": "#9e9ac8",
    "R2": "#807dba",
    "RTE": "#6a51a3",
    "R1": "#cbc9e2",
    "L2": "#756bb1",
    "Dong-R4": "#54278f",
    "Dualen": "#3f007d",
    "CRE": "#bcbddc",
    "Tad1": "#9e9ac8",
    "Rex-Babar": "#4a1486",
    "Proto2": "#6a51a3",
    "Proto1": "#807dba",
 
    # SINE group: greens
    "tRNA": "#74c476",         # old SINE2/tRNA
    "5S": "#238b45",           # old SINE3/5S
    "7SL": "#bae4b3",          # old SINE1/7SL
    "U": "#edf8e9",
 
    # RC / Helitron
    "Helitron": "#e41a1c",
 
    # Empty / non-superfamily groups
    "No superfamily": "gray",
}
 
order_colors_new = {
    "LTR": "#377eb8",
    "DNA": "#ff7f00",           # old TIR color
    "LINE": "#984ea3",
    "SINE": "#4daf4a",
    "RC": "#e41a1c",            # old Helitron color
    "PLE": "#a65628",
 
    # New broad categories
    "Satellite": "#999999",
    "Structural_RNA": "#66c2a5",
    "Other": "#bdbdbd",
}